# Where to stimulate the brain? — Control vs. strength

Main analysis notebook (Topic 7).

- **Hypothesis A (Stiso et al., 2019):** network control theory (average controllability, impulse response) predicts the effects of stimulation.
- **Hypothesis B (Tu et al., 2018):** controllability adds little beyond simple network statistics (node strength).

**Team:** Melike Hatun Bayır, Yeluripati Srividya  
Each section lists its owner. Fill in your own sections; keep decisions in the *Decision log* at the end.

**Working rules:** clear outputs before committing (`Kernel → Restart & Clear Output`) to avoid merge conflicts; move reusable functions to `src/`.

## 0. Setup

In [1]:
# Imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `src` can be imported

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import (
    load_ccep, runs, electrodes_tsv, channels_tsv,
    connectome_files, load_connectome, load_lut,
    FIG_DIR, TABLE_DIR, CACHE_DIR, PROCESSED_DIR,
)

### 0.1 Analysis parameters
All choices that are varied in the robustness analysis (Section 5) live here.

In [2]:
# Parameters (main analysis; robustness changes one line at a time)
EXCLUSION_RADIUS_MM = 13                  # main: 13 mm | robustness: 10, 20
SOZ_HANDLING = "all_patients"             # main: "all_patients" | robustness: "annotated_only"
CURRENT_HANDLING = "all_with_covariate"   # main: "all_with_covariate" | robustness: "8mA_only"
DROP_MIXED_CURRENT_PAIRS = True           # pairs with two currents in one run are dropped
RESPONSE_RULE = None         # when does a region count as responding? (1.5)
CONNECTOME_TRANSFORM = None  # raw / log(1 + w) (1.6)
CONSISTENCY_THRESHOLD = None # keep connections present in >= x% of subjects (1.6)
K_HORIZON = None             # time horizon K
N_NULL = None                # number of null networks
SEED = 0

---
# 1. Data

## 1.1 Load CCEP data

**Owner:** _TBD_  
**Status:** not started

Load `ccepData_V2.mat`, inspect the structure (patients, runs, `n1_peak_sample`, `tt`, `stimpair_currents`, `nativeElecDistances`).

 Overview of the patients

In [3]:
cc = load_ccep()
overview = pd.DataFrame([{
    "patient": p["id"], "session": p["ses"], "age": p["age"],
    "runs": len(runs(p)),
    "channels": len(runs(p)[0]["channel_names"]),
    "stim_pairs": sum(len(r["stimpair_names"]) for r in runs(p)),
} for p in cc])
print(len(cc), "patients |", overview["runs"].sum(), "runs |",
      overview["stim_pairs"].sum(), "stimulation pairs (pair x run)")
overview.describe().round(1)

74 patients | 116 runs | 4907 stimulation pairs (pair x run)


,age,runs,channels,stim_pairs
count,74.0,74.0,74.0,74.0
mean,21.2,1.6,116.4,66.3
std,12.9,1.1,26.2,25.6
min,4.0,1.0,64.0,39.0
25%,11.2,1.0,98.0,51.2
50%,17.0,1.0,133.0,57.0
75%,30.0,2.0,133.0,73.0
max,51.0,8.0,133.0,222.0


Structure of one patient and one run

In [4]:
p0 = cc[0]; r0 = runs(p0)[0]
print("patient fields:", sorted(p0.keys()))
print("run fields:", sorted(r0.keys()))
print("n1_peak_sample:", r0["n1_peak_sample"].shape, "(channels x stimulation pairs; NaN = no N1)")
print("tt:", r0["tt"].shape, f"from {r0['tt'][0]:.2f} s to {r0['tt'][-1]:.2f} s, step {np.diff(r0['tt'][:2])[0]*1000:.2f} ms")
print("nativeElecDistances:", p0["nativeElecDistances"].shape, "(mm, order = electrodes.tsv)")
print("example pair:", r0["stimpair_names"][0], "| current (A):", r0["stimpair_currents"][0])

patient fields: ['age', 'electrodes', 'id', 'nativeElecDistances', 'nrRuns', 'rois', 'run', 'ses']
run fields: ['channel_DestrieuxLabel', 'channel_DestrieuxNr', 'channel_names', 'good_channels', 'n1_peak_sample', 'runName', 'stimpair_DestrieuxLabel', 'stimpair_DestrieuxNr', 'stimpair_currents', 'stimpair_names', 'tt']
n1_peak_sample: (133, 46) (channels x stimulation pairs; NaN = no N1)
tt: (2560,) from -2.00 s to 3.00 s, step 1.95 ms
nativeElecDistances: (133, 133) (mm, order = electrodes.tsv)
example pair: PT01-PT02 | current (A): 0.007


## 1.2 Electrode and channel information

**Owner:** _TBD_  
**Status:** not started

Read `electrodes.tsv` and `channels.tsv` per patient; match them to the channels and stimulation pairs in the `.mat` file.

All electrodes in one table

In [5]:
E = pd.concat([electrodes_tsv(p["id"], p["ses"]).assign(patient=p["id"]) for p in cc], ignore_index=True)
print(E.shape)
for col in ["group", "soz", "resected"]:
    print(col, E[col].value_counts(dropna=False).to_dict())
# group "other" = non-brain channels (ECG, EMG, markers): no label -> removed by the no_label rule

(8655, 16)
group {'grid': 4410, 'other': 3123, 'strip': 920, 'depth': 202}
soz {'no': 8302, 'yes': 353}
resected {'no': 7999, 'yes': 656}


Do the .mat channels match electrodes.tsv?

In [6]:
rows = []
for p in cc:
    el = electrodes_tsv(p["id"], p["ses"])
    for r in runs(p):
        names = list(r["channel_names"])
        ch = channels_tsv(p["id"], p["ses"], r["runName"])
        rows.append(dict(
            patient=p["id"], run=r["runName"],
            same_order_as_tsv=names == list(el["name"][:len(names)]),
            channels_missing_in_tsv=len(set(names) - set(el["name"])),
            bad_channels=int((ch.set_index("name").reindex(names)["status"] != "good").sum()),
        ))
chk = pd.DataFrame(rows)
print("runs where channel order differs from electrodes.tsv:", (~chk["same_order_as_tsv"]).sum(),
      "in", chk.loc[~chk["same_order_as_tsv"], "patient"].nunique(), "patients -> always match by NAME")
print("channels not found in electrodes.tsv:", chk["channels_missing_in_tsv"].sum())
print("bad channels per run: median", chk["bad_channels"].median())

runs where channel order differs from electrodes.tsv: 13 in 8 patients -> always match by NAME
channels not found in electrodes.tsv: 203
bad channels per run: median 36.5


## 1.3 Exclusions

**Owner:** _TBD_  
**Status:** not started

Apply, and count the effect of, each rule:
- responses within the exclusion radius of a stimulated electrode (`nativeElecDistances`)
- depth electrodes
- no hemisphere / atlas label
- bad channels (`channels.tsv`)
- resected electrodes
- seizure onset zone (only part of the patients annotated → decision)
- same rules for the stimulated electrodes? (decision)

Apply the exclusion rules (recording and stimulated electrodes)

In [7]:
def electrode_flags(el, names):
    """One row per channel: which exclusion rules apply (electrode-level rules)."""
    e = el.reindex(names)
    nr = pd.to_numeric(e["Destrieux_label"], errors="coerce")
    return pd.DataFrame({
        "depth":    e["group"].eq("depth").values,
        "no_label": ~((nr > 0) & e["hemisphere"].isin(["L", "R"])).values,
        "resected": e["resected"].eq("yes").values,
        "soz":      e["soz"].eq("yes").values,
    }, index=names)

def near_stimulation(names, a, b, pos, row_idx, D):
    """True for channels closer than EXCLUSION_RADIUS_MM to stimulated electrode a or b,
    and for a and b themselves. D = nativeElecDistances, in electrodes.tsv order."""
    if a in pos and b in pos:
        d = np.where(row_idx >= 0, D[np.clip(row_idx, 0, None)][:, [pos[a], pos[b]]].min(axis=1), np.nan)
    else:                                       # stimulated electrode missing in electrodes.tsv
        d = np.full(len(names), np.nan)
    near = ~(d >= EXCLUSION_RADIUS_MM)          # unknown distance counts as "too near"
    near[[names.index(x) for x in (a, b) if x in names]] = True
    return near

soz_annotated = E.groupby("patient")["soz"].apply(lambda s: s.eq("yes").any())
print("patients with SOZ annotation:", soz_annotated.sum(), "of", len(soz_annotated))

ELECTRODE_RULES = ["depth", "no_label", "bad_channel", "resected", "soz"]
pair_rows, resp_rows = [], []
for p in cc:
    if SOZ_HANDLING == "annotated_only" and not soz_annotated[p["id"]]:
        continue
    el = electrodes_tsv(p["id"], p["ses"]).set_index("name")
    D = p["nativeElecDistances"]
    pos = {n: i for i, n in enumerate(el.index)}
    for r in runs(p):
        names = list(r["channel_names"])
        fl = electrode_flags(el, names)
        st = channels_tsv(p["id"], p["ses"], r["runName"]).set_index("name").reindex(names)["status"]
        fl["bad_channel"] = (st != "good").values
        rec_ok = ~fl[ELECTRODE_RULES].any(axis=1).values
        row_idx = np.array([pos.get(n, -1) for n in names])
        n1, tt = r["n1_peak_sample"], r["tt"]
        cur = r["stimpair_currents"]
        if cur.dtype != object:
            cur = list(cur)
        for j, pair in enumerate(r["stimpair_names"]):
            a, b = pair.split("-")
            mA = np.round(np.ravel(cur[j]) * 1000, 1)
            stim_fl = fl.reindex([a, b]).fillna(True)        # unknown electrode = flagged
            row = dict(patient=p["id"], run=r["runName"], pair=pair,
                       unique_pair=p["id"] + "|" + "-".join(sorted([a, b])),
                       mA=mA[0] if mA.size == 1 else np.nan,
                       mixed_current=mA.size > 1)
            for rule in ELECTRODE_RULES:
                row["stim_" + rule] = bool(stim_fl[rule].any())
            row["stim_ok"] = not any(row["stim_" + rr] for rr in ELECTRODE_RULES)
            near = near_stimulation(names, a, b, pos, row_idx, D)
            has = ~np.isnan(n1[:, j])
            row["rec_total"] = len(names)
            row["rec_kept"] = int((rec_ok & ~near).sum())
            row["n1_total"] = int(has.sum())
            row["n1_kept"] = int((has & rec_ok & ~near).sum())
            pair_rows.append(row)
            if row["stim_ok"]:
                for i in np.where(rec_ok & ~near)[0]:
                    resp_rows.append((p["id"], r["runName"], pair, names[i], bool(has[i]),
                                      tt[int(n1[i, j]) - 1] * 1000 if has[i] else np.nan))

pairs = pd.DataFrame(pair_rows)
responses = pd.DataFrame(resp_rows, columns=["patient", "run", "pair", "channel", "n1", "latency_ms"])
print(len(pairs), "stimulation pairs (pair x run) |", len(responses), "kept recording entries")

patients with SOZ annotation: 36 of 74
4907 stimulation pairs (pair x run) | 230811 kept recording entries


How much does each rule remove? (recording side, in this order)

In [8]:
rec_counts = []
for p in cc:
    if SOZ_HANDLING == "annotated_only" and not soz_annotated[p["id"]]:
        continue
    el = electrodes_tsv(p["id"], p["ses"]).set_index("name")
    D = p["nativeElecDistances"]; pos = {n: i for i, n in enumerate(el.index)}
    for r in runs(p):
        names = list(r["channel_names"])
        fl = electrode_flags(el, names)
        st = channels_tsv(p["id"], p["ses"], r["runName"]).set_index("name").reindex(names)["status"]
        fl["bad_channel"] = (st != "good").values
        row_idx = np.array([pos.get(n, -1) for n in names])
        n1 = r["n1_peak_sample"]
        for j, pair in enumerate(r["stimpair_names"]):
            a, b = pair.split("-")
            near = near_stimulation(names, a, b, pos, row_idx, D)
            has = ~np.isnan(n1[:, j])
            alive = np.ones(len(names), bool)
            steps = [(rr, fl[rr].values) for rr in ELECTRODE_RULES] + \
                    [(f"within {EXCLUSION_RADIUS_MM} mm / stimulated itself", near)]
            for rule, mask in steps:
                removed = alive & mask
                rec_counts.append((rule, removed.sum(), (removed & has).sum()))
                alive &= ~mask

summary = pd.DataFrame(rec_counts, columns=["rule", "entries_removed", "n1_removed"]).groupby("rule", sort=False).sum()
total_entries, total_n1 = pairs["rec_total"].sum(), pairs["n1_total"].sum()
summary["entries_%"] = (100 * summary["entries_removed"] / total_entries).round(1)
summary["n1_%"] = (100 * summary["n1_removed"] / total_n1).round(1)
print(f"recording entries: {total_entries} | N1 responses: {total_n1}")
summary

recording entries: 577740 | N1 responses: 40168


,entries_removed,n1_removed,entries_%,n1_%
rule,,,,
depth,11288,0,2.0,0.0
no_label,194660,90,33.7,0.2
bad_channel,25096,0,4.3,0.0
resected,33543,4129,5.8,10.3
soz,6810,948,1.2,2.4
within 13 mm / stimulated itself,28513,13022,4.9,32.4


Stimulated electrodes: how many pairs are dropped?

In [9]:
stim_summary = pd.Series({rule: pairs["stim_" + rule].sum() for rule in ELECTRODE_RULES},
                         name="pairs with a flagged stimulated electrode")
print("pairs kept (no flagged stimulated electrode):", pairs["stim_ok"].sum(), "of", len(pairs))
stim_summary

pairs kept (no flagged stimulated electrode): 3872 of 4907


depth          101
no_label        30
bad_channel    238
resected       690
soz            415
Name: pairs with a flagged stimulated electrode, dtype: int64

## 1.4 Stimulation current

**Owner:** _TBD_  
**Status:** not started

8 mA vs. 4 mA near sensorimotor cortex (`stimpair_currents`): decide how to handle (covariate, exclude, or stratify).

In [10]:
kept = pairs[pairs["stim_ok"]].copy()
if DROP_MIXED_CURRENT_PAIRS:
    print("mixed-current pairs dropped:", kept["mixed_current"].sum())
    kept = kept[~kept["mixed_current"]]
if CURRENT_HANDLING == "8mA_only":
    kept = kept[kept["mA"] == 8]
kept["weaker_pulse"] = (kept["mA"] < 8).astype(int)       # covariate for the analysis

cur_tab = kept.groupby("mA").agg(measurements=("pair", "size"),
                                 unique_pairs=("unique_pair", "nunique"),
                                 patients=("patient", "nunique"))
cur_tab["share_%"] = (100 * cur_tab["measurements"] / len(kept)).round(1)
print("final:", len(kept), "measurements |", kept["unique_pair"].nunique(), "unique pairs |",
      kept["patient"].nunique(), "patients")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
kept.to_csv(PROCESSED_DIR / "stim_pairs_clean.csv", index=False)
resp_kept = responses.merge(kept[["patient", "run", "pair"]], on=["patient", "run", "pair"])
resp_kept.to_csv(PROCESSED_DIR / "responses_clean.csv.gz", index=False)
print("saved:", len(kept), "pairs and", len(resp_kept), "recording entries")
cur_tab

mixed-current pairs dropped: 5
final: 3867 measurements | 3467 unique pairs | 74 patients
saved: 3867 pairs and 230463 recording entries


,measurements,unique_pairs,patients,share_%
mA,,,,
4.0,234,230,12,6.1
6.0,43,43,4,1.1
7.0,204,81,1,5.3
8.0,3386,3120,73,87.6


## 1.5 Electrode → region assignment

**Owner:** _TBD_  
**Status:** not started

Map each electrode to a connectome node via `Destrieux_label` + `hemisphere`. Define the stimulation site (region(s) of the two stimulated electrodes) and the rule for when a recorded region counts as responding.

## 1.6 Group connectome

**Owner:** _TBD_  
**Status:** not started

Load the 50 MICA-MICs matrices and the LUT:
- matrix format (upper triangle, subcortical rows, medial wall)
- node selection
- weight transform and averaging
- consistency threshold
- final symmetric matrix `A` (document every step)

## 1.7 Observed spread and response patterns

**Owner:** _TBD_  
**Status:** not started

For each stimulation site: proportion of recorded regions with an N1 (optionally N1 latencies), and the binary response pattern across recorded regions.

## 1.8 Dataset description

**Owner:** _TBD_  
**Status:** not started

Patients (age, number of runs), stimulation sites, brain coverage per patient, distribution of the observed spread. Figures for the presentation.

---
# 2. Models

## 2.1 Linear network model

**Owner:** _TBD_  
**Status:** not started

$x_{k+1} = \tilde A x_k + e_j u_k$, with $\tilde A = A / (1 + \lambda_{\max}(A))$.
- why the normalisation makes the system stable
- how to model a stimulation pair whose electrodes lie in two regions

## 2.2 Average controllability

**Owner:** _TBD_  
**Status:** not started

Gramian $W_j = \sum_{k=0}^{K-1} \tilde A^k e_j e_j^\top (\tilde A^k)^\top$, $AC_j = \mathrm{trace}(W_j)$.
Show/verify that $AC_j = \sum_{k=0}^{K-1} \|\tilde A^k e_j\|^2$.

## 2.3 Predicted response patterns

**Owner:** _TBD_  
**Status:** not started

- impulse response (incl. indirect paths): $\sum_{k=1}^{K} \tilde A^k e_j$
- direct connections only: $\tilde A e_j$

## 2.4 Simple network measures

**Owner:** _TBD_  
**Status:** not started

Node strength $d_j = \sum_i A_{ij}$.

## 2.5 Null networks

**Owner:** _TBD_  
**Status:** not started

Strength-preserving randomised connectomes (e.g. `bctpy`); recompute all measures. Optional: spatial null / distance-only prediction.

## 2.6 Numerical aspects

**Owner:** _TBD_  
**Status:** not started

Condition number of the Gramian for different K and normalisations; correlation of $AC_j$ and $d_j$ across regions; which measures are affected by conditioning (trace vs. inverse, e.g. minimum control energy).

---
# 3. Analysis

## 3.1 Observed spread vs. AC and strength

**Owner:** _TBD_  
**Status:** not started

Does controllability explain variance beyond strength? Account for sites nested in patients (per-patient analyses or mixed model); covariates: stimulation current, number of recorded regions.

## 3.2 Response patterns: observed vs. predicted

**Owner:** _TBD_  
**Status:** not started

Per site: correlate the observed pattern with the impulse-response prediction and the direct-connection prediction. Note: connectome is symmetric, responses are directed.

## 3.3 Comparison with null networks

**Owner:** _TBD_  
**Status:** not started

Same predictions from the null networks; where do the real-connectome results fall in the null distribution?

## 3.4 AC vs. strength and Gramian conditioning

**Owner:** _TBD_  
**Status:** not started

Report the relation between $AC_j$ and $d_j$ and the numerical conditioning.

---
# 4. Results figures
Final, presentation/report-ready plots (labelled axes, legends, readable fonts, error bars). Save to `FIG_DIR`.

## 4.1 Electrode coverage and spread

**Owner:** _TBD_  
**Status:** not started

Which regions are covered; how stimulation spreads.

## 4.2 Spread vs. controllability and strength

**Owner:** _TBD_  
**Status:** not started



## 4.3 Predicted vs. observed response patterns + null networks

**Owner:** _TBD_  
**Status:** not started



## 4.4 AC–strength relation and conditioning

**Owner:** _TBD_  
**Status:** not started



---
# 5. Robustness

## 5.1 Robustness analyses

**Owner:** _TBD_  
**Status:** not started

Re-run the key results while varying:
- K
- normalisation
- exclusion radius
- connectome processing (transform, threshold, nodes)
- electrode → region assignment / response rule

## 5.2 Robustness summary

**Owner:** _TBD_  
**Status:** not started

One table/figure summarising how the main results change.

---
# Decision log
| # | Decision | Options | Chosen | Justification | Who |
|---|---|---|---|---|---|
| 1 | SOZ handling for unannotated patients | | | | |
| 2 | Exclusion rules for stimulated electrodes | | | | |
| 3 | Stimulation current (8 vs 4 mA) | | | | |
| 4 | Response rule for a region | | | | |
| 5 | Connectome nodes | | | | |
| 6 | Weight transform | | | | |
| 7 | Consistency threshold | | | | |
| 8 | Stimulation pair in two regions | | | | |
| 9 | Nesting in patients (model) | | | | |

# Task ownership
| Section | Owner |
|---|---|
| 1.1–1.4 Data loading & exclusions | |
| 1.5–1.7 Region assignment, connectome, spread | |
| 1.8 Dataset description | |
| 2.1–2.4 Models & measures | |
| 2.5 Null networks | |
| 2.6 Numerical aspects | |
| 3.1 Spread vs. AC/strength | |
| 3.2–3.3 Response patterns & nulls | |
| 3.4 AC–strength & conditioning | |
| 4 Figures | |
| 5 Robustness | |